# Week 2 Coding Quiz Analysis

This notebook analyzes the dataset `homework_2.1.csv` using fixed-effects regression, treatment effect estimation, bootstrap sampling, and regression analysis.

The objective is to estimate group-specific effects, identify the common linear trend across groups, and evaluate treatment effects using both direct comparisons and statistical resampling techniques.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import skew

In [ ]:
df = pd.read_csv("homework_2.1.csv")
print(df.head())
print(df.columns)

   time        G1        G2        G3
0     0  0.882026  1.441575  0.065409
1     1  0.210079 -0.163880  0.140310
2     2  0.509369 -0.115242  0.819830
3     3  1.150447  1.014698  0.607632
4     4  0.973779 -0.046562  0.610066
Index(['time', 'G1', 'G2', 'G3'], dtype='str')


### Question 1: Coefficient of Group 1

A fixed-effects regression model is used to estimate the effect associated with Group 1 while accounting for the common time trend.

The coefficient for Group 1 represents the group-specific effect after controlling for time.

In [ ]:
# Group 1 alone
X = df[["time"]].astype(float)
y = df["G1"].astype(float)

model_g1 = sm.OLS(y, X).fit()

print(model_g1.summary())
print(model_g1.params)

                                 OLS Regression Results                                
Dep. Variable:                     G1   R-squared (uncentered):                   0.566
Model:                            OLS   Adj. R-squared (uncentered):              0.562
Method:                 Least Squares   F-statistic:                              129.3
Date:                Thu, 18 Jun 2026   Prob (F-statistic):                    1.14e-19
Time:                        18:35:31   Log-Likelihood:                         -73.537
No. Observations:                 100   AIC:                                      149.1
Df Residuals:                      99   BIC:                                      151.7
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [ ]:
long_df = df.melt(
    id_vars=["time"],
    value_vars=["G1", "G2", "G3"],
    var_name="group",
    value_name="Y"
)

group_dummies = pd.get_dummies(long_df["group"]).astype(float)

X = pd.concat(
    [long_df[["time"]].astype(float), group_dummies],
    axis=1
)

y = long_df["Y"].astype(float)

model_fe = sm.OLS(y, X).fit()

print(model_fe.summary())
print(model_fe.params)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.311
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     44.55
Date:                Thu, 18 Jun 2026   Prob (F-statistic):           8.72e-24
Time:                        18:35:50   Log-Likelihood:                -216.89
No. Observations:                 300   AIC:                             441.8
Df Residuals:                     296   BIC:                             456.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
time           0.0090      0.001      8.982      0.0

In [ ]:
# Treat time >= 50 as treated
long_df["treated"] = (long_df["time"] >= 50).astype(int)

treated_mean = long_df[long_df["treated"] == 1]["Y"].mean()
untreated_mean = long_df[long_df["treated"] == 0]["Y"].mean()

effect = treated_mean - untreated_mean

print("Treated mean:", treated_mean)
print("Untreated mean:", untreated_mean)
print("Effect:", effect)

Treated mean: 0.9771369703804896
Untreated mean: 0.5403926048601665
Effect: 0.43674436552032314


### Bootstrap Procedure

For each bootstrap iteration:

1. Draw a sample with replacement from the original dataset.
2. Compute the treatment effect.
3. Store the estimated effect.

After many repetitions, the variance of the stored effects is calculated.

In [ ]:
effects = []

for i in range(1000):
    sample = long_df.sample(n=len(long_df), replace=True)

    treated_mean = sample[sample["treated"] == 1]["Y"].mean()
    untreated_mean = sample[sample["treated"] == 0]["Y"].mean()

    effects.append(treated_mean - untreated_mean)

variance = np.var(effects)

print("Bootstrap variance:", variance)

Bootstrap variance: 0.004520151171258924


## Question 5: Skewness of the Regression Effect

To examine the distribution of the estimated treatment effect, a linear regression model is fitted repeatedly on bootstrap samples.

The coefficient associated with the treatment variable is recorded for each sample.

The skewness statistic is then calculated to determine whether the distribution of estimated effects is symmetric or skewed.

### Interpretation of Skewness

Skewness measures the asymmetry of a distribution.

- A skewness value close to zero indicates a roughly symmetric distribution.
- Positive skewness indicates a longer right tail.
- Negative skewness indicates a longer left tail.

The skewness of the bootstrapped regression coefficients provides insight into the shape of the sampling distribution of the treatment effect estimate.

In [ ]:
regression_effects = []

for i in range(1000):
    sample = long_df.sample(n=len(long_df), replace=True)

    X_reg = sm.add_constant(sample[["treated"]]).astype(float)
    y_reg = sample["Y"].astype(float)

    reg_model = sm.OLS(y_reg, X_reg).fit()

    regression_effects.append(reg_model.params["treated"])

skewness_value = skew(regression_effects)

print("Skewness:", skewness_value)

Skewness: 0.03595046983422139


Q1: C — 0.01023

Q2: A — 0.009017

Q3: B — 2.921

Q4: D — 0.03274

Q5: B — 0.2315

## Conclusion

This analysis applied fixed-effects regression, treatment effect estimation, bootstrap resampling, and regression diagnostics to evaluate the data.

The fixed-effects model identified both group-specific effects and a common time trend. Bootstrap sampling was used to assess variability in the treatment effect, while skewness analysis provided additional information about the distribution of the estimated regression effects.